In [3]:
import os

In [1]:
%pwd

'c:\\Users\\kaush\\Desktop\\PROJECTS\\Chicken-disease-classification\\chicken-disease-classification-project-end2end\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\kaush\\Desktop\\PROJECTS\\Chicken-disease-classification\\chicken-disease-classification-project-end2end'

In [6]:
import tensorflow as tf
from pathlib import Path


In [7]:
def build_model(input_shape, num_classes, learning_rate):
    base_model = tf.keras.applications.ConvNeXtTiny(
        input_shape=input_shape,
        weights=None,
        include_top=False
    )

    for layer in base_model.layers:
        layer.trainable = False

    x = tf.keras.layers.Flatten()(base_model.output)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs=base_model.input, outputs=outputs)

    model.compile(
        optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [9]:
from cnnClassifier.utils.common import read_yaml, create_directories, save_json
from cnnClassifier.constants import *

params = read_yaml(PARAMS_FILE_PATH)
config =  read_yaml(CONFIG_FILE_PATH)

[2026-07-21 00:31:31,296: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-21 00:31:31,314: INFO: common: yaml file: config\config.yaml loaded successfully]


In [11]:
from dataclasses import dataclass
@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int

In [12]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path, config: EvaluationConfig):

        model = build_model(
            input_shape=tuple(config.params_image_size),
            num_classes=2,   # Change if you have more classes
            learning_rate=config.all_params.LEARNING_RATE
        )

        model.load_weights(path)

        return model

    def evaluation(self):

        self.model = self.load_model(
            self.config.path_of_model,
            self.config
        )

        self._valid_generator()

        self.score = self.model.evaluate(self.valid_generator)

    def save_score(self):

        scores = {
            "loss": self.score[0],
            "accuracy": self.score[1]
        }

        save_json(
            path=Path("scores.json"),
            data=scores
        )

In [ ]:
@staticmethod
def load_model(path: Path):

    model = build_model(
        input_shape=(224,224,3),
        num_classes=2,           # change accordingly
        learning_rate=0.001      # from params
    )

    model.load_weights(path)

    return model

In [13]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    path_of_model: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int

In [14]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):
        
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_validation_config(self) -> EvaluationConfig:
        eval_config = EvaluationConfig(
            path_of_model= self.config.training.training_model_path,
            training_data=  Path(self.config.data_ingestion.unzip_dir) / "Chicken-fecal-images",
            all_params= self.params,
            params_image_size= self.params.IMAGE_SIZE,
            params_batch_size= self.params.BATCH_SIZE
        )
        return eval_config

In [15]:
from urllib.parse import urlparse

In [20]:
class Evaluation:
    def __init__(self, config: EvaluationConfig):
        self.config = config

    
    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.30
        )

        dataflow_kwargs = dict(
            target_size=self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    
    @staticmethod
    def load_model(path: Path) -> tf.keras.Model:
        return tf.keras.models.load_model(path)
    

    def evaluation(self):
        self.model = self.load_model(self.config.path_of_model)
        self._valid_generator()
        self.score = model.evaluate(self.valid_generator)

    
    def save_score(self):
        scores = {"loss": self.score[0], "accuracy": self.score[1]}
        save_json(path=Path("scores.json"), data=scores)


In [16]:
try:
    config = ConfigurationManager()
    val_config = config.get_validation_config()
    evaluation = Evaluation(val_config)
    evaluation.evaluation()
    evaluation.save_score()

except Exception as e:
   raise e

[2026-07-21 00:38:41,484: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-21 00:38:41,490: INFO: common: yaml file: params.yaml loaded successfully]

[2026-07-21 00:38:42,365: WARNING: module_wrapper: From c:\Users\kaush\miniconda3\envs\chicken-env\lib\site-packages\keras\src\backend\tensorflow\core.py:232: The name tf.placeholder is deprecated. Please use tf.compat.v1.placeholder instead.
]
[2026-07-21 00:38:43,347: WARNING: config: TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]
Found 116 images belonging to 2 classes.
8/8 ━━━━━━━━━━━━━━━━━━━━ 43s 5s/step - accuracy: 0.5000 - loss: 77.0060
[2026-07-21 00:39:28,774: INFO: common: json file saved at: scores.json]
